# Churn model — demo notebook

This notebook ships with **Notebook Context**. Focus any code cell and press `Ctrl+Alt+C` (or click 🧠 in the cell toolbar) to capture why the code is the way it is.

CTX-002 below was captured *before* `days_since_last_order` was added, so it shows as ⚠ code changed — open the Notebook Context timeline in the sidebar and click it to see the diff.

<!-- notebook-context
@context_id: CTX-004
@type: experiment
@created_timestamp: 2026-09-19 10:57:37 -05:00
@updated_timestamp: 2026-09-19 10:59:08 -05:00
@notebook_path: examples/churn_demo.ipynb
@cell_id: dca73fd6
@cell_index: 3
@cell_hash: sha256:e3b0c44298fc
-->

### 🧪 Experiment · CTX-004

**Hypothesis:** —
**Change:** —
**Result:** —
**Decision:** —
<sub>🧠 CTX-004 · created 2026-09-19 10:57:37 -05:00 · linked to cell 2</sub>

<!-- notebook-context
@context_id: CTX-005
@type: data-context
@created_timestamp: 2026-09-19 10:57:49 -05:00
@updated_timestamp: 2026-09-19 10:57:49 -05:00
@notebook_path: examples/churn_demo.ipynb
@cell_id: dca73fd6
@cell_index: 3
@cell_hash: sha256:e3b0c44298fc
-->

### 🗂️ Data Context · CTX-005

**Dataset:** _Source table / file / API, plus version or snapshot date_

**Grain:** _One row = ? (e.g. one customer per calendar month)_

**Date range:** _Period covered, and whether the last period is complete_

**Filters:** _Rows excluded and why (test accounts, nulls, outliers)_

**Known limitations:** _Missing values, leakage risk, sampling bias, stale dimensions_

<sub>🧠 CTX-005 · created 2026-09-19 10:57:49 -05:00 · linked to cell 3</sub>

<!-- notebook-context
@context_id: CTX-001
@type: data-context
@created_timestamp: 2026-09-12 09:14:05 +05:30
@updated_timestamp: 2026-09-19 10:59:53 -05:00
@notebook_path: examples/churn_demo.ipynb
@cell_index: 4
@cell_hash: sha256:a888c455a0ff
-->

### 🗂️ Data Context · CTX-001

**Dataset:** `data/orders_2025Q1.parquet`, snapshot taken 2026-04-02

**Grain:** one customer × calendar month

**Date range:** 2025-01-01 → 2025-03-31 (March is complete)

**Filters:** dropped `account_type == "test"` (~1.2% of rows)

**Known limitations:** refunds are not netted out of `amount`; no rows for customers with zero orders in a month

<sub>🧠 CTX-001 · created 2026-09-12 09:14:05 +05:30 · linked to cell 2</sub>

In [ ]:
import pandas as pd

orders = pd.read_parquet('data/orders_2025Q1.parquet')
orders = orders[orders.account_type != 'test']
monthly = orders.groupby(['customer_id', orders.order_ts.dt.to_period('M')]).agg(
    n_orders=('order_id', 'count'), revenue=('amount', 'sum')
).reset_index()

<!-- notebook-context
@context_id: CTX-002
@type: experiment
@created_timestamp: 2026-09-15 16:40:51 +05:30
@updated_timestamp: 2026-09-15 16:40:51 +05:30
@notebook_path: examples/churn_demo.ipynb
@cell_index: 4
@cell_hash: sha256:7dfd102101d4
-->

### 🧪 Experiment · CTX-002

**Hypothesis:** Recency matters more than volume for churn — a days-since-last-order feature should lift AUC

**Change:** baseline uses only `n_orders` and `revenue`

**Result:** —

**Decision:** —

<sub>🧠 CTX-002 · created 2026-09-15 16:40:51 +05:30 · linked to cell 4</sub>

In [ ]:
monthly['days_since_last_order'] = (
    monthly.order_ts.dt.end_time - orders.groupby('customer_id').order_ts.max()
).dt.days
X = monthly[['n_orders', 'revenue', 'days_since_last_order']]
y = monthly['churned_next_month']

<!-- notebook-context
@context_id: CTX-003
@type: decision
@created_timestamp: 2026-09-16 11:02:37 +05:30
@updated_timestamp: 2026-09-19 11:01:18 -05:00
@notebook_path: examples/churn_demo.ipynb
@cell_index: 8
@cell_hash: sha256:4c5b058743ec
-->

### ⚖️ Decision · CTX-003

**Decision:** Use LightGBM instead of logistic regression for the v1 model v3

**Why:** non-linear interaction between revenue and recency; LR plateaued at AUC 0.71

**Evidence:** LR 0.71 vs LGBM 0.78 on the same 20% holdout (seed 42)

**Alternatives:** XGBoost (similar AUC, slower to train); LR with manual interaction terms (brittle)

**Revisit if:** the model needs to be explainable to Finance, or training data grows past ~10M rows

<sub>🧠 CTX-003 · created 2026-09-16 11:02:37 +05:30 · linked to cell 6</sub>

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
clf = LGBMClassifier(n_estimators=300, learning_rate=0.05).fit(X_tr, y_tr)
print('AUC', roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1]))

<!-- notebook-context
@context_id: CTX-006
@type: note
@created_timestamp: 2026-09-19 11:01:32 -05:00
@updated_timestamp: 2026-09-19 11:03:39 -05:00
@notebook_path: examples/churn_demo.ipynb
@cell_id: 97cd25f4
@cell_index: 11
@cell_hash: sha256:e3b0c44298fc
-->

### 📝 General Note · CTX-006

**Note:** _Whatever future-you or a teammate needs to know about this cell_ this sis amadadsajd

**Follow-up:** —
<sub>🧠 CTX-006 · created 2026-09-19 11:01:32 -05:00 · linked to cell 10</sub>

<!-- notebook-context
@context_id: CTX-007
@type: result
@created_timestamp: 2026-09-19 11:02:08 -05:00
@updated_timestamp: 2026-09-19 11:02:08 -05:00
@notebook_path: examples/churn_demo.ipynb
@cell_id: 97cd25f4
@cell_index: 11
@cell_hash: sha256:e3b0c44298fc
-->

### 📈 Result · CTX-007

**Metric:** _What was measured, on which split / period_

**Value:** _The number(s), with the baseline you compared against_

**Interpretation:** _What this means for the question you're actually answering_

**Caveats:** _Why this could mislead: leakage, small n, cherry-picked window, seed luck_

**Next step:** _What this result makes you do next_

<sub>🧠 CTX-007 · created 2026-09-19 11:02:08 -05:00 · linked to cell 11</sub>

<!-- notebook-context
@context_id: CTX-008
@type: data-context
@created_timestamp: 2026-09-19 11:05:52 -05:00
@updated_timestamp: 2026-09-19 11:05:52 -05:00
@notebook_path: examples/churn_demo.ipynb
@cell_id: 97cd25f4
@cell_index: 12
@cell_hash: sha256:f7683d40c1af
-->

### 🗂️ Data Context · CTX-008

**Dataset:** _Source table / file / API, plus version or snapshot date_

**Grain:** _One row = ? (e.g. one customer per calendar month)_

**Date range:** _Period covered, and whether the last period is complete_

**Filters:** _Rows excluded and why (test accounts, nulls, outliers)_

**Known limitations:** _Missing values, leakage risk, sampling bias, stale dimensions_

<sub>🧠 CTX-008 · created 2026-09-19 11:05:52 -05:00 · linked to cell 12</sub>

In [2]:
justin = 3